# SME Post-check — rà soát tín dụng sau phê duyệt

Bốn cell: nhập liệu → cấu hình → chạy → ghi kết quả.
**Một lần chạy là một hồ sơ.** Không có chế độ chạy theo lô.

In [ ]:
import sys
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env", override=True)

TESTCASE_ID    = "case_demo"
TAX_CODE       = "0101234567"
APPROVAL_DATE  = "2025-01-02"     # ngày lô phê duyệt bắt đầu hiệu lực
POSTCHECK_DATE = "2025-09-14"     # ngày thực hiện rà soát

CASE_DIR = PROJECT_ROOT / "samples" / TESTCASE_ID

print(f"{TESTCASE_ID} · MST {TAX_CODE} · rà soát ngày {POSTCHECK_DATE}")

In [ ]:
from src.config import Config, build_llm

config = Config(
    financial_statement_llm = build_llm("MODEL_FINANCIAL_STATEMENT"),
    proposal_llm            = build_llm("MODEL_PROPOSAL"),
    sitevisit_llm           = build_llm("MODEL_SITEVISIT"),
    cic_s10a_llm            = build_llm("MODEL_CIC_S10A"),
    cic_r20_llm             = build_llm("MODEL_CIC_R20"),
    commentary_llm          = build_llm("MODEL_COMMENTARY"),

    query_executor = None,

    max_extraction_calls = 40,
)

In [ ]:
from src.pipeline import run_postcheck

result = run_postcheck(
    case_dir       = CASE_DIR,
    config         = config,
    tax_code       = TAX_CODE,
    approval_date  = APPROVAL_DATE,
    postcheck_date = POSTCHECK_DATE,
)

print("Tài liệu:", len(result.documents))
print("Kết quả:", result.counts)
for finding in result.findings:
    # if finding.status != "PASS":
    print(f"  {finding.status:18s} {finding.rule_id}  {finding.title}")

In [ ]:
# --- Cell 4: write the run out ---
import json
from datetime import datetime

# run_postcheck already rendered the report, commentary included.
run_dir = PROJECT_ROOT / "logs" / f"{TESTCASE_ID}_{datetime.now():%Y%m%d_%H%M%S}"
run_dir.mkdir(parents=True, exist_ok=True)
(run_dir / "report.md").write_text(result.report_markdown, encoding="utf-8")
(run_dir / "result.json").write_text(
    json.dumps(result.to_dict(), ensure_ascii=False, indent=2, default=str),
    encoding="utf-8")

try:
    import markdown as markdown_lib
    from weasyprint import HTML
    from src.utils.report.report_style import REPORT_CSS, tag_wide_tables
    body = markdown_lib.markdown(result.report_markdown,
                                 extensions=["tables", "fenced_code"])
    html = ('<html><head><meta charset="utf-8"><style>' + REPORT_CSS
            + "</style></head><body>" + tag_wide_tables(body) + "</body></html>")
    HTML(string=html).write_pdf(run_dir / "report.pdf")
    print("đã ghi report.pdf")
except Exception as exc:
    print(f"bỏ qua PDF ({type(exc).__name__}: {exc}); report.md vẫn đầy đủ")

print("kết quả tại:", run_dir)
